## State Replication Machine System Analytics
This notebook visualizes the low-latency order-flow simulation results generated by `command_stress`.

This notebook reads directly from the real, exact data output files generated by the C++ stress test. No fake or random data is generated here; it parses the mathematically correct metrics produced during the simulation.

In [ ]:
!pip install -q pandas plotly

In [ ]:
import os
import json
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

import plotly.io as pio
pio.templates.default = "plotly_dark"

artifact_dir = './'
print(f"Fetching data from: {artifact_dir}")

### 1. Throughput Timeline & Backlog

In [ ]:
timeline_path = os.path.join(artifact_dir, 'timeline.csv')
if os.path.exists(timeline_path):
    df_t = pd.read_csv(timeline_path)
    fig = make_subplots(specs=[[{"secondary_y": True}]])
    fig.add_trace(go.Scatter(x=df_t['elapsed_ms'], y=df_t['interval_mps'], name='Interval MPS (Millions)', mode='lines', line=dict(color='#00ffcc')), secondary_y=False)
    fig.add_trace(go.Scatter(x=df_t['elapsed_ms'], y=df_t['moving_avg_mps'], name='Moving Avg MPS', mode='lines', line=dict(color='#ff00ff')), secondary_y=False)
    fig.add_trace(go.Scatter(x=df_t['elapsed_ms'], y=df_t['backlog'], name='Backlog', mode='lines', line=dict(color='#ff9900', dash='dot')), secondary_y=True)
    fig.update_layout(title='System Throughput & Queue Backlog Over Time', xaxis_title='Elapsed (ms)')
    fig.update_yaxes(title_text='Througput (MPS)', secondary_y=False)
    fig.update_yaxes(title_text='Queue Backlog', secondary_y=True)
    fig.show()
else:
    print("timeline.csv not found")

### 2. High-Resolution Latency Profile

In [ ]:
summary_path = os.path.join(artifact_dir, 'summary.json')
if os.path.exists(summary_path):
    with open(summary_path, 'r') as f:
        summary = json.load(f)
    lat = summary.get('latency', {})
    percentiles = ['p50', 'p75', 'p90', 'p95', 'p99', 'p99.9', 'max']
    keys = ['p50_ns', 'p75_ns', 'p90_ns', 'p95_ns', 'p99_ns', 'p999_ns', 'max_ns']
    values = [lat.get(k, 0) / 1000.0 for k in keys]  # Convert to microseconds
    
    fig = px.bar(x=percentiles, y=values, text=[f"{v:.1f}µs" for v in values], 
                 title='Deterministic Latency Distribution (µs)', 
                 labels={'x': 'Percentile', 'y': 'Latency (µs)'}, 
                 color=values, color_continuous_scale='Turbo')
    fig.update_traces(textposition='outside')
    fig.show()
    
    print("\n--- Stress Test Core Metrics ---")
    print(f"Total Messages: {summary['throughput'].get('messages', 0):,}")
    print(f"Mean Latency: {lat.get('mean_ns', 0) / 1000.0:.2f} µs")
    print(f"Throughput Jitter: {summary['throughput'].get('throughput_jitter_mps', 0):.4f} MPS")
else:
    print("summary.json not found")

### 3. Raw Event Flow & PnL Analysis

In [ ]:
events_path = os.path.join(artifact_dir, 'raw_events.jsonl')
if os.path.exists(events_path):
    print("Parsing up to 100,000 raw market events for visualization...")
    events = []
    with open(events_path, 'r') as f:
        for i, line in enumerate(f):
            if i > 100000: break
            try:
                events.append(json.loads(line))
            except:
                pass
    df = pd.DataFrame(events)
    # Convert data
    for col in ['px', 'qty', 'latency_ns', 'observed_ns']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')
            
    if 'type' in df.columns:
        fig1 = px.pie(df, names='type', title='Order Event Distribution', hole=0.4, color_discrete_sequence=px.colors.sequential.Plasma)
        fig1.show()
        
    # Latency Scatter (detecting stress regions)
    df_ok = df[df['latency_ns'].notna() & (df['latency_ns'] > 0)].copy()
    if not df_ok.empty:
        df_ok['latency_us'] = df_ok['latency_ns'] / 1000.0
        df_sample = df_ok.sample(min(15000, len(df_ok)))
        fig2 = px.scatter(df_sample, x='observed_ns', y='latency_us', color='type', 
                          title='Latency Microburst Analysis (Scatter)', opacity=0.7)
        fig2.show()
        
    # PnL & Asset volume
    if 'asset' in df.columns and 'qty' in df.columns:
        vol_df = df.groupby('asset')['qty'].sum().reset_index()
        fig3 = px.bar(vol_df, x='asset', y='qty', title='Global Traded Volume per Asset', color='qty', color_continuous_scale='Viridis')
        fig3.show()
else:
    print("raw_events.jsonl not found.")